In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

INKAR_CSV = "20251215.csv"
PSY_CSV = "Psychotherapeuten.csv"

def to_numeric(s: pd.Series) -> pd.Series:
    # robust: "1.234,56" -> 1234.56
    return pd.to_numeric(
        s.astype(str)
         .str.replace(".", "", regex=False)
         .str.replace(",", ".", regex=False),
        errors="coerce"
    )

def to_ags_from_kennziffer(x) -> str | None:
    # Kennziffer kommt bei dir teils als float rein (z.B. 9471.0) -> "09471"
    try:
        if pd.isna(x):
            return None
        return str(int(float(x))).zfill(5)
    except Exception:
        return None

# =========================
# 1) CSVs laden (Semikolon!)
# =========================
inkar = pd.read_csv(INKAR_CSV, sep=";", encoding="utf-8")
psy = pd.read_csv(PSY_CSV, sep=";", encoding="utf-8")

# =========================
# 2) Schlüssel bauen (AGS)
# =========================
inkar["AGS"] = inkar["Kennziffer"].map(to_ags_from_kennziffer)

psy = psy[psy["EBENE"] == "PB"].copy()  # Planungsbereiche (entspricht deinen Kreisen/Städten)
psy["AGS"] = (
    psy["AGS"].astype(str)
      .str.replace(r"\.0$", "", regex=True)
      .str.replace(r"\D", "", regex=True)
      .str.zfill(5)
)

# =========================
# 3) Merge
# =========================
df = inkar.merge(psy, on="AGS", how="inner", suffixes=("_inkar", "_psy"))
if df.empty:
    raise ValueError("Merge ergab 0 Zeilen. Prüfe AGS/Kennziffer oder Schreibweise/Filter.")

# =========================
# 4) Variablen vorbereiten
# =========================
# X für Korrelation A:
df["einwohnerdichte"] = to_numeric(df["Einwohnerdichte"])

# Zentralität für Korrelation B:
df["ober_pct"] = to_numeric(df["Bevölkerung in Oberzentren"])
df["mittel_pct"] = to_numeric(df["Bevölkerung in Mittelzentren"])
df["zentral_score"] = 2 * df["ober_pct"] + 1 * df["mittel_pct"]  # frei wählbar, aber gut interpretierbar

# Y (Therapieplätze/Versorgung) – du kannst eins davon wählen:
df["kassensitze"] = to_numeric(df["Kassensitze"])  # absolute Plätze
df["kass_100k"] = to_numeric(df["Kassensitz/100.000Einwohner"])  # pro 100k (für Korrelation oft besser)
df["versorgungsgrad"] = to_numeric(df["Versorgungsgrad"])  # Prozent

# =========================
# 5) Korrelationen rechnen
# =========================
def corr(xcol, ycol, label):
    tmp = df[[xcol, ycol]].dropna()
    r, p = pearsonr(tmp[xcol], tmp[ycol])
    print(f"\n{label}")
    print(f"N = {len(tmp)} | r = {r:.3f} | p = {p:.6f}")
    return tmp, r, p

# Empfehlung: nimm Y = kass_100k ODER versorgungsgrad
tmpA, rA, pA = corr("einwohnerdichte", "kass_100k", "Korrelation A: Einwohnerdichte ↔ Kassensitze pro 100.000")
tmpB, rB, pB = corr("zentral_score", "kass_100k", "Korrelation B: Zentralität ↔ Kassensitze pro 100.000")

# =========================
# 6) Plots
# =========================
plt.figure()
plt.scatter(tmpA["einwohnerdichte"], tmpA["kass_100k"])
plt.xlabel("Einwohnerdichte (Einw./km²)")
plt.ylabel("Kassensitze pro 100.000 Einwohner")
plt.title("Korrelation A")
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure()
plt.scatter(tmpB["zentral_score"], tmpB["kass_100k"])
plt.xlabel("Zentralitäts-Score (2*Oberzentren + Mittelzentren)")
plt.ylabel("Kassensitze pro 100.000 Einwohner")
plt.title("Korrelation B")
plt.grid(True)
plt.tight_layout()
plt.show()

# =========================
# 7) Export für Kontrolle
# =========================
df[[
    "AGS", "Raumeinheit", "EBENE_NAME",
    "einwohnerdichte", "ober_pct", "mittel_pct", "zentral_score",
    "kassensitze", "kass_100k", "versorgungsgrad"
]].to_csv("merged_nordbayern.csv", index=False)

print("\nExportiert: merged_nordbayern.csv")

FileNotFoundError: [Errno 2] No such file or directory: '20251215.csv'